# Name That Model - Enhanced Training Pipeline

Advanced stylometric features and ensemble models for AI authorship identification.

In [1]:
import pandas as pd
import numpy as np
import re
import html
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

Libraries loaded


## 1. Load and Preprocess Data

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Training samples: {len(train)}')
print(f'Test samples: {len(test)}')

def normalize_text(t):
    if pd.isna(t):
        return ''
    t = str(t)
    t = html.unescape(t)
    t = t.replace('\u2019', "'").replace('\u2018', "'").replace('\u201c', '"').replace('\u201d', '"')
    t = t.replace('\u2013', '-').replace('\u2014', '--').replace('\u2026', '...')
    t = t.replace('\ufffd', '')
    t = re.sub(r'[\r\n\t]+', ' ', t)
    t = re.sub(r'\s+', ' ', t).strip()
    return t

train['text_norm'] = train['text'].apply(normalize_text)
test['text_norm'] = test['text'].apply(normalize_text)

if 'model' in train.columns:
    print(f'Number of classes: {train["model"].nunique()}')
else:
    print('ERROR: model column not found')

print('Data preprocessing complete')

Training samples: 2699
Test samples: 3
Number of classes: 18
Data preprocessing complete


## 2. Build Enhanced Pipeline

In [3]:
char_vectorizer = TfidfVectorizer(
    analyzer='char',
    ngram_range=(2, 5),
    max_features=15000,
    lowercase=False,
    sublinear_tf=True
)

word_vectorizer = TfidfVectorizer(
    analyzer='word',
    ngram_range=(1, 3),
    max_features=10000,
    stop_words=None,
    lowercase=True,
    sublinear_tf=True,
    min_df=2,
    max_df=0.95
)

punct_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=8000,
    lowercase=True,
    sublinear_tf=True
)

from sklearn.pipeline import FeatureUnion, Pipeline
text_features = FeatureUnion([
    ('char', char_vectorizer),
    ('word', word_vectorizer),
    ('punct', punct_vectorizer)
])

clf1 = LogisticRegression(
    max_iter=3000,
    solver='saga',
    C=2.0,
    class_weight='balanced',
    random_state=42,
)

clf2 = CalibratedClassifierCV(
    LinearSVC(C=1.0, class_weight='balanced', max_iter=3000, random_state=123),
    method='sigmoid',
    cv=3
)

clf3 = LogisticRegression(
    max_iter=3000,
    solver='lbfgs',
    C=3.0,
    class_weight='balanced',
    random_state=789,
)

ensemble = VotingClassifier(
    estimators=[('lr1', clf1), ('svc', clf2), ('lr2', clf3)],
    voting='soft',
    weights=[3, 2, 3]
)

pipeline = Pipeline([
    ('tfidf', text_features),
    ('clf', ensemble)
])

print('Enhanced pipeline defined')

Enhanced pipeline defined


## 3. Train Model

In [4]:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(train['model'])

print('Training ensemble classifier...')
pipeline.fit(train['text_norm'].astype(str), encoded_labels)
print('Training complete')

Training ensemble classifier...


Training complete


## 4. Evaluate

In [5]:
train_preds = pipeline.predict(train['text_norm'].astype(str))
bal_acc = balanced_accuracy_score(encoded_labels, train_preds)
acc = accuracy_score(encoded_labels, train_preds)
print(f'Accuracy: {acc:.4f}')
print(f'Balanced accuracy: {bal_acc:.4f}')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, train['text_norm'].astype(str), encoded_labels, cv=cv, scoring='balanced_accuracy')
print(f'Cross-validated balanced accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

Accuracy: 1.0000
Balanced accuracy: 1.0000


Cross-validated balanced accuracy: 0.1533 (+/- 0.0089)


## 5. Predict and Create Submission

In [6]:
test_preds = pipeline.predict(test['text_norm'].astype(str))
test_labels = label_encoder.inverse_transform(test_preds)

valid_models = [
    'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
    'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
    'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
    'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'
]
submission = pd.DataFrame({
    'id': test['id'].astype(str),
    'model': test_labels
})
submission['model'] = submission['model'].apply(lambda x: x if x in valid_models else 'gpt-5.5')
submission = submission.drop_duplicates(subset=['id'], keep='first')
submission = submission.dropna(subset=['id', 'model'])
submission.to_parquet('submission.parquet', index=False)
print('Submission created: submission.parquet')
print(submission.shape)
print(submission.to_string(index=False))

Submission created: submission.parquet
(3, 2)
          id            model
00353fb1c541 gemini-3.8-flash
005f5a9a6139      gpt-5.6-sol
0062a53d5487 gemini-3.7-flash
